In [8]:
# ----------------- Inference & Evaluation -----------------
def run_inference(model, tokenizer, dataset, max_len=128, num_samples=100):
    model.eval()
    model.to("cuda" if torch.cuda.is_available() else "cpu")
    tokenizer.src_lang = "kk"
    
    bleu = evaluate.load("sacrebleu")
    chrf = evaluate.load("chrf")

    preds = []
    refs = []
    source=[]

    for example in tqdm(dataset.select(range(num_samples)), desc="🔹 Running inference"):
        inputs = tokenizer(example["kazakh"], return_tensors="pt", truncation=True, padding=True, max_length=max_len).to(model.device)
        with torch.no_grad():
            generated = model.generate(**inputs, forced_bos_token_id=tokenizer.get_lang_id("ru"), max_length=max_len)
        pred = tokenizer.decode(generated[0], skip_special_tokens=True)
        ref = example["russian"]

        preds.append(pred)
        refs.append([ref])  # BLEU expects list of references
        source.append(example["kazakh"])

        bleu.add(prediction=pred, reference=[ref])
        chrf.add(prediction=pred, reference=ref)

    bleu_score = bleu.compute()
    chrf_score = chrf.compute()

    print("🔹 BLEU:", bleu_score)
    print("🔹 chrF++:", chrf_score)

    return preds, refs, source

In [2]:
from train import load_data

import os
import pandas as pd
import polars as pl
import torch
import evaluate
from datasets import Dataset
import numpy as np
from transformers import (
    M2M100Tokenizer,
    M2M100ForConditionalGeneration,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments
)
from tqdm import tqdm

model_dir='/home/lilo/experiments/exp_m2m100_065_kz_rus'
test='/home/lilo/cleaned_data/test_dedup.txt'

test_set = load_data(test, 0.65)
tokenizer = M2M100Tokenizer.from_pretrained(model_dir)
model = M2M100ForConditionalGeneration.from_pretrained(model_dir)


/usr/local/lib/python3.10/dist-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


🔹 Using GPU: 0 (NVIDIA GeForce RTX 3090)


In [3]:
model.device

device(type='cpu')

In [9]:
from train import tokenize_data

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model=model.to(device)
dataset = tokenize_data(test_set, tokenizer, max_length=112, cache_dir='./cache')

preds, ref, source = run_inference(model, tokenizer, dataset, max_len=112, num_samples=1000) ###set len(test_set) for full metrics

🔹 Tokenizing dataset with cache file: ./cache/tokenized_cache.arrow


🔹 Running inference: 100%|██████████| 1000/1000 [03:51<00:00,  4.33it/s]


🔹 BLEU: {'score': 30.743320011339637, 'counts': [7995, 4859, 3390, 2457], 'totals': [13814, 12814, 11852, 10974], 'precisions': [57.87606775734762, 37.919463087248324, 28.602767465406682, 22.38928376161837], 'bp': 0.8928909122158043, 'sys_len': 13814, 'ref_len': 15379}
🔹 chrF++: {'score': 51.863807137906804, 'char_order': 6, 'word_order': 0, 'beta': 2}


In [5]:
len(preds)

1000

In [24]:
# i=679
print(ref[i][0], '\n', preds[i], '\n', source[i])
i+=1

(X)Выдача лицензии на импорт и (или) экспорт отдельных видов товаров 
 (X)Выдача лицензии на импорт и (или) экспорт отдельных видов товаров 
 (X)Жекелеген тауарлар түрлерінің импортына және (немесе) экспортына лицензия беру


In [27]:
# i=679
print(ref[i][0], '\n', preds[i], '\n', source[i])
i+=1

Главная > Перечень Продуктов > Солнечный Свет > Солнечный Уличный Свет > 22LED солнечной энергии двойной голова человека индукции лампы 
 Главная > Перечень Продуктов > Солнечный Свет > Солнечный Уличный Свет > 22 светодиодные солнечные батареи индукционные лампы для двух человек 
 Басты > Өнімдер > Күн сәулесі > Solar Street Light > 22LED күн батареясы екі адамға арналған индукциялық шам


filtered res

In [29]:
from train import load_data

import os
import pandas as pd
import polars as pl
import torch
import evaluate
from datasets import Dataset
import numpy as np
from transformers import (
    M2M100Tokenizer,
    M2M100ForConditionalGeneration,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments
)
from tqdm import tqdm

model_dir='/home/lilo/experiments/exp_m2m100_filtered065_kz_rus'
test='/home/lilo/cleaned_data/test_dedup.txt'

test_set = load_data(test, 0.65)
tokenizer = M2M100Tokenizer.from_pretrained(model_dir)
model = M2M100ForConditionalGeneration.from_pretrained(model_dir)


In [31]:
from train import tokenize_data

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model=model.to(device)
dataset = tokenize_data(test_set, tokenizer, max_length=112)

predsf, reff, sourcef = run_inference(model, tokenizer, dataset, max_len=112, num_samples=1000) ###set len(test_set) for full metrics

🔹 Tokenizing dataset with cache file: ./cache/tokenized_cache.arrow


🔹 Running inference: 100%|██████████| 1000/1000 [03:51<00:00,  4.31it/s]


🔹 BLEU: {'score': 30.743320011339637, 'counts': [7995, 4859, 3390, 2457], 'totals': [13814, 12814, 11852, 10974], 'precisions': [57.87606775734762, 37.919463087248324, 28.602767465406682, 22.38928376161837], 'bp': 0.8928909122158043, 'sys_len': 13814, 'ref_len': 15379}
🔹 chrF++: {'score': 51.863807137906804, 'char_order': 6, 'word_order': 0, 'beta': 2}
